# Regression Inference Pipeline
**Input:** Any CSV with a `smiles` column + trained models  
**Output:** `submission.csv` with predicted `homo` and `lumo`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator
from sklearn.impute import SimpleImputer

print('All imports OK')

In [ ]:
# ── INPUT ──────────────────────────────────────────────
INPUT_FILE  = '25262_regression_sample_submission.csv'  # <-- change path
HOMO_MODEL  = 'homo_model.joblib'
LUMO_MODEL  = 'lumo_model.joblib'
OUTPUT_FILE = 'submission.csv'
# ───────────────────────────────────────────────────────

df = pd.read_csv(INPUT_FILE)
assert 'smiles' in df.columns, "CSV must contain a 'smiles' column"
print(f'Loaded {len(df)} molecules from {INPUT_FILE}')

In [ ]:
# Load models and feature lists
homo_model    = joblib.load(HOMO_MODEL)
homo_features = joblib.load('homo_features.joblib')
homo_rdkit    = joblib.load('homo_rdkit_cols.joblib')

lumo_model    = joblib.load(LUMO_MODEL)
lumo_features = joblib.load('lumo_features.joblib')
lumo_rdkit    = joblib.load('lumo_rdkit_cols.joblib')

print(f'HOMO model loaded | features: {len(homo_features)}')
print(f'LUMO model loaded | features: {len(lumo_features)}')

In [ ]:
def compute_features(smiles_list, rdkit_cols):
    """Compute RDKit descriptors + Morgan fingerprints (same as training)."""
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(rdkit_cols)
    gen_r2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    gen_r3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)

    desc_rows, fp_r2_rows, fp_r3_rows = [], [], []
    failed = 0

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            desc_rows.append([np.nan] * len(rdkit_cols))
            fp_r2_rows.append([0] * 2048)
            fp_r3_rows.append([0] * 2048)
            failed += 1
        else:
            desc_rows.append(list(calculator.CalcDescriptors(mol)))
            fp_r2 = np.zeros(2048, dtype=np.float32)
            for idx, count in gen_r2.GetCountFingerprint(mol).GetNonzeroElements().items():
                fp_r2[idx % 2048] = count
            fp_r2_rows.append(fp_r2)

            fp_r3 = np.zeros(2048, dtype=np.float32)
            for idx, count in gen_r3.GetCountFingerprint(mol).GetNonzeroElements().items():
                fp_r3[idx % 2048] = count
            fp_r3_rows.append(fp_r3)

    if failed:
        print(f'Warning: {failed} SMILES could not be parsed')

    df_desc = pd.DataFrame(desc_rows, columns=rdkit_cols)
    df_fp_r2 = pd.DataFrame(fp_r2_rows, columns=[f'morgan_r2_{i}' for i in range(2048)])
    df_fp_r3 = pd.DataFrame(fp_r3_rows, columns=[f'morgan_r3_{i}' for i in range(2048)])

    result = pd.concat([df_desc, df_fp_r2, df_fp_r3], axis=1)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result

print('Computing features...')
all_features = compute_features(df['smiles'].tolist(), homo_rdkit)
print(f'Done. Shape: {all_features.shape}')

In [ ]:
def prepare_X(features_df, feature_cols):
    """Select correct features, impute, clip extreme values."""
    X = features_df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.values.astype(np.float64)
    X[~np.isfinite(X)] = 0.0
    X = np.clip(X, -1e10, 1e10)
    return X

# HOMO
X_homo = prepare_X(all_features, homo_features)
homo_preds = homo_model.predict(X_homo)

# LUMO — recompute if rdkit cols differ
if homo_rdkit != lumo_rdkit:
    all_features_lumo = compute_features(df['smiles'].tolist(), lumo_rdkit)
else:
    all_features_lumo = all_features

X_lumo = prepare_X(all_features_lumo, lumo_features)
lumo_preds = lumo_model.predict(X_lumo)

print(f'HOMO | min: {homo_preds.min():.4f}  max: {homo_preds.max():.4f}  mean: {homo_preds.mean():.4f}')
print(f'LUMO | min: {lumo_preds.min():.4f}  max: {lumo_preds.max():.4f}  mean: {lumo_preds.mean():.4f}')

In [ ]:
submission = pd.DataFrame({
    'smiles': df['smiles'],
    'homo':   homo_preds,
    'lumo':   lumo_preds
})

submission.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {OUTPUT_FILE}')
print(submission.head())